# CoolPrompt metrics tutorial

### Quick setup

In [ ]:
import os
from coolprompt.assistant import PromptTuner

os.environ["OPENAI_API_KEY"] = "sk-proj-..."

prompt_tuner = PromptTuner()

CoolPrompt supports 2 task types and a set of metrics: 
- `classification` - a common classification
    - `accuracy`
    - `f1` (f1-macro)
- `generation` - a general new text generation
    - `bleu`
    - `rouge`
    - `meteor`
    - `bertscore`
    - `codebertscore` (BertScore for coding assignments)
    - `geval` 
    - `em` (exact match for math tasks)
    - `llm_as_judge`

For each run you can define as follows:

## Classification metrics

### Accuracy

In [ ]:
prompt_tuner.run(
    start_prompt="Classify a text sentiment",
    task="classification",
    metric="accuracy",
)

### F1-score

In [ ]:
prompt_tuner.run(
    start_prompt="Classify a sentence sentiment",
    task="classification",
    metric="f1",
)

## Generation metrics

### BLEU

In [ ]:
prompt_tuner.run(
    start_prompt="Summarize the text",
    task="generation",
    metric="bleu",
)

### ROUGE-L

In [ ]:
prompt_tuner.run(
    start_prompt="Summarize the text",
    task="generation",
    metric="rouge",
)

### METEOR

In [ ]:
prompt_tuner.run(
    start_prompt="Summarize the text",
    task="generation",
    metric="meteor",
)

### BertScore

In [ ]:
prompt_tuner.run(
    start_prompt="Summarize the text",
    task="generation",
    metric="bertscore",
)

### CodeBertScore

In [ ]:
prompt_tuner.run(
    start_prompt="Use description to generate code",
    task="generation",
    metric="codebertscore",
)

### EM (ExactMatch)
suitable for numeric answers

In [ ]:
prompt_tuner.run(
    start_prompt="Solve the equation",
    task="generation",
    metric="em",
)

### Deepeval GEval metric 

GEval is an LLM-as-a-judge metric that checks whether a generated answer matches the expected output according to natural-language instructions. 

You can pass either a single `geval_criteria` string (as in the example below) or a list of explicit `evaluation_steps`, and optionally select which fields the judge sees via `evaluation_params` (default INPUT, ACTUAL_OUTPUT, EXPECTED_OUTPUT).

In [ ]:
from datasets import load_dataset

samsum = load_dataset("knkarthick/samsum")
dataset = samsum["train"]["dialogue"][:40]
targets = samsum["train"]["summary"][:40]

geval_criteria = (
    "Evaluate whether the assistant's summary is an accurate, concise, and coherent "
    "paraphrase of the original text. The summary should capture all key facts and "
    "important events from the expected output, avoid adding new information that is "
    "not supported by the original text, and omit minor irrelevant details. "
)
res = prompt_tuner.run(
    start_prompt="Summarize the text",
    task="generation",
    dataset=dataset,
    target=targets,
    method="hyper_light",
    metric="geval",
    geval_criteria=geval_criteria,
    verbose=2
)

print("Initial GEval score:", prompt_tuner.init_metric)
print("Final GEval score:", prompt_tuner.final_metric)
print("Final prompt", res)

Using evaluation steps example

In [ ]:
geval_steps = [
    "1. Compare the assistant's summary with the reference and list the key facts each one mentions.",
    "2. Check whether the assistant introduces any unsupported information or misses essential events.",
    "3. Decide if the assistant's summary is concise and coherent while covering all required details.",
]

final_prompt = prompt_tuner.run(
    start_prompt="Summarize the text",
    task="generation",
    dataset=dataset,
    target=targets,
    method="hyper_light",
    metric="geval",
    geval_evaluation_steps=geval_steps,
    verbose=2,
 )

print("Initial GEval score:", prompt_tuner.init_metric)
print("Final GEval score:", prompt_tuner.final_metric)
print("Final prompt:", final_prompt)

# LLM as judge mertic #


There are 4 available templates for llm as judge metric:

- Accuracy: The answer is factually correct and contains no mistakes. It uses the right terms and numbers and does not introduce wrong information.

- Coherence: The answer is well structured and easy to follow. 
Ideas are in a logical order and there are no contradictions.

- Fluency: The answer is written in natural, correct language. 
Grammar, vocabulary and phrasing are clear and do not make the text hard to read.

- Relevance: The answer directly responds to the request and stays on topic. 
It does not add unnecessary or unrelated details. 


In [ ]:
result = prompt_tuner.run(
    start_prompt="Summarize the text",
    task="generation",
    dataset=dataset,
    target=targets,
    method="hyper_light",
    metric="llm_as_judge",
    llm_as_judge_criteria="relevance",
    generate_num_samples=20,
    verbose=2,
)

result2 = prompt_tuner.run(
    start_prompt="Summarize the text",
    task="generation",
    dataset=dataset,
    target=targets,
    method="hyper_light",
    metric="llm_as_judge",
    llm_as_judge_criteria=["relevance", "coherence"],
    generate_num_samples=20,
    verbose=2,
)


custom = {
    "creativity": """You will be given a response to a question.
    Rate the creativity on a scale from 1 to {metric_ceil}.
    Return ONLY a single number.

    Source: {request}
    Response: {response}

    Creativity score (number only):"""
}

result3 = prompt_tuner.run(
    start_prompt="Summarize the text",
    task="generation",
    dataset=dataset,
    target=targets,
    method="hyper_light",
    metric="llm_as_judge",
    llm_as_judge_criteria=["creativity"],
    llm_as_judge_custom_templates=custom,
    generate_num_samples=20,
    verbose=2,
)